# 01 - Khám phá & tiền xử lý dữ liệu (Phase 1)

Đề tài 4: VAE vs GAN vs Diffusion. Notebook này kiểm tra dữ liệu sau tiền xử lý trước khi train.

In [ ]:
# Colab: bỏ comment
# !git clone https://github.com/Kienhtth1/DeepLearning_BTL.git
# %cd DeepLearning_BTL
# !pip install -q -r requirements.txt
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import matplotlib.pyplot as plt
import torch
from torchvision.utils import make_grid
from src.data import get_dataloader, denormalize
from src.utils import set_seed, load_config

set_seed(42)
cfg = load_config("configs/base.yaml")   # đổi sang configs/celeba.yaml nếu dùng CelebA
DATASET = cfg["data"]["name"]
loader, info = get_dataloader(DATASET, split="train", batch_size=64, num_workers=0,
                              image_size=cfg["data"]["image_size"])
info

## 1. Ảnh mẫu sau tiền xử lý

In [ ]:
x, y = next(iter(loader))
print("shape:", tuple(x.shape), "| min/max:", x.min().item(), x.max().item())
grid = make_grid(denormalize(x), nrow=8, padding=2)
plt.figure(figsize=(8, 8)); plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray"); plt.axis("off"); plt.show()

## 2. Mỗi lớp một hàng (nếu có nhãn)

In [ ]:
if info.num_classes:
    per_class = {c: [] for c in range(info.num_classes)}
    for xb, yb in loader:
        for img, lab in zip(xb, yb.tolist()):
            if len(per_class[lab]) < 8:
                per_class[lab].append(img)
        if all(len(v) == 8 for v in per_class.values()):
            break
    fig, axes = plt.subplots(info.num_classes, 1, figsize=(8, 1.1 * info.num_classes))
    for c, ax in enumerate(axes):
        g = make_grid(denormalize(torch.stack(per_class[c])), nrow=8, padding=1)
        ax.imshow(g.permute(1, 2, 0).squeeze(), cmap="gray"); ax.set_axis_off()
        ax.set_title(info.class_names[c], fontsize=8, loc="left")
    plt.tight_layout(); plt.show()

## 3. Phân bố lớp & histogram giá trị pixel

In [ ]:
from collections import Counter
labels = Counter()
pixels = []
for i, (xb, yb) in enumerate(loader):
    labels.update(yb.tolist())
    if i < 20:
        pixels.append(xb.flatten())
pixels = torch.cat(pixels)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
if info.num_classes:
    names = info.class_names
    ax[0].bar([names[k] for k in sorted(labels)], [labels[k] for k in sorted(labels)])
    ax[0].tick_params(axis="x", rotation=45); ax[0].set_title("Số ảnh mỗi lớp (train)")
ax[1].hist(pixels.numpy(), bins=50); ax[1].set_title("Histogram pixel sau chuẩn hoá [-1, 1]")
plt.tight_layout(); plt.show()

## 4. Nhận xét (điền sau khi chạy)

- Số ảnh train/val/test: ...
- Cân bằng lớp: ...
- Lý do chọn kích thước ảnh: ...
- Lưu ý cho Phase 2: ...